# 01 - Khảo sát Dữ liệu
## Data Exploration

Mục tiêu: Phân tích chi tiết cấu trúc, phân bố và đặc trưng của 2 dataset:
- **Khan Academy YouTube Dataset** (~52,000 videos)
- **Coursera Courses Dataset** (~41,000 courses)

## 1.1 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Tao thu muc luu anh
os.makedirs('../report/images', exist_ok=True)
os.makedirs('../data/raw', exist_ok=True)

plt.style.use('seaborn-v0_8-darkgrid')
print("Libraries loaded successfully!")

## 1.2 Load Khan Academy Dataset

In [ ]:
# Load Khan Academy
khan_df = pd.read_csv('../youtube_khan_academy.csv', low_memory=False)
print(f"Shape: {khan_df.shape}")
print(f"\nColumns ({len(khan_df.columns)}):")
for i, col in enumerate(khan_df.columns):
    print(f"  [{i+1:2d}] {col}: {khan_df[col].dtype}")

In [ ]:
# Xem 5 dong dau
khan_df.head()

In [ ]:
# Thong ke mo ta
khan_df.describe(include='all').T.head(30)

## 1.3 Phân tích Missing Values - Khan Academy

In [ ]:
# Gia tri thieu
missing = khan_df.isnull().sum()
missing_pct = (missing / len(khan_df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
print(f"So cot co gia tri thieu: {len(missing_df)}")
missing_df

In [ ]:
# Bieu do missing values
fig, ax = plt.subplots(figsize=(12, 6))
missing_df.head(20)['Missing %'].plot(kind='barh', ax=ax, color='#4C72B0')
ax.set_title('Missing Values Percentage - Khan Academy', fontsize=14, fontweight='bold')
ax.set_xlabel('Missing %')
plt.tight_layout()
plt.savefig('../report/images/nb01_khan_missing.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.4 Load Coursera Dataset

In [ ]:
# Load Coursera
coursera_df = pd.read_csv('../courses_en.csv', low_memory=False)
print(f"Shape: {coursera_df.shape}")
print(f"\nColumns: {coursera_df.columns.tolist()}")
coursera_df.head()

In [ ]:
# Missing values Coursera
missing_c = coursera_df.isnull().sum()
missing_c_pct = (missing_c / len(coursera_df) * 100).round(2)
missing_c_df = pd.DataFrame({'Count': missing_c, 'Pct': missing_c_pct})
missing_c_df[missing_c_df['Count'] > 0].sort_values('Count', ascending=False)

## 1.5 Phân bố Dữ liệu

In [ ]:
# Histogram - View Count
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Khan Academy Video Statistics', fontsize=16, fontweight='bold')

vc = pd.to_numeric(khan_df['view_count'], errors='coerce').dropna()
vc_filtered = vc[vc < vc.quantile(0.95)]
axes[0].hist(vc_filtered, bins=50, color='#4C72B0', alpha=0.8, edgecolor='white')
axes[0].set_title('View Count Distribution')
axes[0].set_xlabel('View Count')
axes[0].set_ylabel('Frequency')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1000:.0f}K'))

lc = pd.to_numeric(khan_df['like_count'], errors='coerce').dropna()
lc_filtered = lc[lc < lc.quantile(0.95)]
axes[1].hist(lc_filtered, bins=50, color='#DD8452', alpha=0.8, edgecolor='white')
axes[1].set_title('Like Count Distribution')
axes[1].set_xlabel('Like Count')
plt.tight_layout()
plt.savefig('../report/images/nb01_khan_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"View count stats: mean={vc.mean():.0f}, median={vc.median():.0f}, max={vc.max():.0f}")

In [ ]:
# Bar Chart - Coursera Categories
cat_counts = coursera_df['category'].value_counts().head(15)
fig, ax = plt.subplots(figsize=(12, 7))
cat_counts.plot(kind='barh', ax=ax, color=plt.cm.viridis(np.linspace(0, 1, len(cat_counts))))
ax.set_title('Top 15 Coursera Course Categories', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Courses')
for i, v in enumerate(cat_counts.values):
    ax.text(v + 10, i, str(v), va='center')
plt.tight_layout()
plt.savefig('../report/images/nb01_coursera_categories.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Pie Chart - Coursera Language
if 'language' in coursera_df.columns:
    lang_counts = coursera_df['language'].value_counts().head(8)
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.pie(lang_counts.values, labels=lang_counts.index,
           autopct='%1.1f%%', startangle=90,
           colors=plt.cm.Set3(np.linspace(0, 1, len(lang_counts))))
    ax.set_title('Coursera Course Languages', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../report/images/nb01_coursera_languages.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Correlation Heatmap - Khan Academy Numeric Features
numeric_cols = ['view_count', 'like_count', 'comment_count',
                'title_sentiment_polarity', 'desc_sentiment_polarity',
                'desc_gunning_fog', 'desc_flesch_reading_ease',
                'desc_flesch_kincaid_grade']
available = [c for c in numeric_cols if c in khan_df.columns]
corr_data = khan_df[available].apply(pd.to_numeric, errors='coerce').corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_data, dtype=bool))
sns.heatmap(corr_data, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', ax=ax, square=True, linewidths=0.5,
            annot_kws={'size': 9})
ax.set_title('Correlation Heatmap - Khan Academy', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../report/images/nb01_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Readability scores distribution
readability_cols = [c for c in khan_df.columns if 'gunning' in c or 'flesch' in c]
if readability_cols:
    fig, axes = plt.subplots(1, min(3, len(readability_cols)), figsize=(15, 5))
    for i, col in enumerate(readability_cols[:3]):
        data = pd.to_numeric(khan_df[col], errors='coerce').dropna()
        data = data[(data > 0) & (data < 30)]
        axes[i].hist(data, bins=40, color=f'C{i}', alpha=0.8, edgecolor='white')
        axes[i].set_title(col.replace('_', ' ').title())
    plt.suptitle('Readability Score Distributions', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../report/images/nb01_readability.png', dpi=150, bbox_inches='tight')
    plt.show()

## 1.6 Summary

In [ ]:
print("="*60)
print("DATASET SUMMARY")
print("="*60)
print(f"Khan Academy:")
print(f"  - Total videos: {len(khan_df):,}")
print(f"  - Columns: {len(khan_df.columns)}")
print(f"  - Missing cells: {khan_df.isnull().sum().sum():,}")
print(f"  - Duplicates (title): {khan_df['title'].duplicated().sum():,}")
print(f"  - Date range: {khan_df['published_at'].min() if 'published_at' in khan_df.columns else 'N/A'} to {khan_df['published_at'].max() if 'published_at' in khan_df.columns else 'N/A'}")
print()
print(f"Coursera:")
print(f"  - Total courses: {len(coursera_df):,}")
print(f"  - Columns: {len(coursera_df.columns)}")
print(f"  - Missing cells: {coursera_df.isnull().sum().sum():,}")
if 'category' in coursera_df.columns:
    print(f"  - Unique categories: {coursera_df['category'].nunique()}")
print()
print("All charts saved to: ../report/images/")